In [1]:
import numpy as np
import tensorflow as tf
import tensorflow.keras as keras
import matplotlib.pyplot as plt
import pickle

In [2]:
from tensorflow.compat.v1 import ConfigProto
from tensorflow.compat.v1 import InteractiveSession
import os
config=tf.compat.v1.ConfigProto()
config.gpu_options.per_process_gpu_memory_fraction = 0.9
config.gpu_options.allow_growth=True
sess=tf.compat.v1.Session(config=config) 

In [3]:
from load_network import load_ViT
from train import WarmUpCosine, CustomWeightDecaySGD
from save_load import load_noise_if_exists, save_layer_outputs_and_labels, load_layer_outputs_and_labels
from weights_bias_cla import load_wb_if_exists
from network import HierarchicalConvEmbedding, TransformerBlock, AddPositionEmbedding
from evaluate_cla import evalu_stream_main_selected, per_group_ovr_accuracy, evalu_select

In [4]:
with open('data.pkl', 'rb') as f:
    [x_train,y_train,x_test,y_test]=pickle.load(f)
y_train_onehot=tf.keras.utils.to_categorical(y_train,num_classes=4)
y_test_onehot=tf.keras.utils.to_categorical(y_test,num_classes=4)

In [5]:
model=load_ViT()

In [6]:
pred_model = tf.keras.Model(
    inputs=model.get_layer("dense_16").output,
    outputs=model.output
)

In [7]:
l_label = [2,3,4,5,6,7,8,9,10,13]

In [8]:
layer_list = [model.layers[l].name for l in l_label]

In [9]:
NOISE_DIR = "./noise/"
save_layer_outputs_and_labels(model, x_train, y_train, layer_list, save_dir="D:/Data_3/ViT-8/layer_cache/base")
for i in range(10):
    NOISE_I_DIR = NOISE_DIR + str(i)
    x_gauss, x_salt, x_move, x_occ = load_noise_if_exists(NOISE_I_DIR)
    save_layer_outputs_and_labels(model, x_gauss, y_train, layer_list, save_dir="D:/Data_3/ViT-8/layer_cache/gauss/"+str(i))
    save_layer_outputs_and_labels(model, x_salt, y_train, layer_list, save_dir="D:/Data_3/ViT-8/layer_cache/salt/"+str(i))

[SAVE] Generating layer outputs for: D:/Data_3/ViT-8/layer_cache/base
[Saved] add_position_embedding: outputs (20000, 8192), labels (20000, 1)
[Saved] transformer_block: outputs (20000, 8192), labels (20000, 1)
[Saved] transformer_block_1: outputs (20000, 8192), labels (20000, 1)
[Saved] transformer_block_2: outputs (20000, 8192), labels (20000, 1)
[Saved] transformer_block_3: outputs (20000, 8192), labels (20000, 1)
[Saved] transformer_block_4: outputs (20000, 8192), labels (20000, 1)
[Saved] transformer_block_5: outputs (20000, 8192), labels (20000, 1)
[Saved] transformer_block_6: outputs (20000, 8192), labels (20000, 1)
[Saved] transformer_block_7: outputs (20000, 8192), labels (20000, 1)
[Saved] dense_16: outputs (20000, 128), labels (20000, 1)
[SAVE] Generating layer outputs for: D:/Data_3/ViT-8/layer_cache/gauss/0
[Saved] add_position_embedding: outputs (20000, 8192), labels (20000, 1)
[Saved] transformer_block: outputs (20000, 8192), labels (20000, 1)
[Saved] transformer_block_1

In [10]:
CACHE_DIR = "./ViT-8/w_and_b_cache"

In [11]:
eva_w,eva_b = load_wb_if_exists(y_train, layer_list, CACHE_DIR, save_dir="D:/Data_3/ViT-8/layer_cache/base")


==== split 0 ====

==== split 1 ====

==== split 2 ====

==== split 3 ====
xi>=0 count: 14693
xi>=0 count: 13958
xi>=0 count: 13844
xi>=0 count: 14139

==== split 0 ====

==== split 1 ====

==== split 2 ====

==== split 3 ====
xi>=0 count: 15289
xi>=0 count: 15098
xi>=0 count: 14843
xi>=0 count: 15470

==== split 0 ====

==== split 1 ====

==== split 2 ====

==== split 3 ====
xi>=0 count: 15776
xi>=0 count: 15741
xi>=0 count: 15380
xi>=0 count: 16091

==== split 0 ====

==== split 1 ====

==== split 2 ====

==== split 3 ====
xi>=0 count: 15763
xi>=0 count: 15903
xi>=0 count: 16432
xi>=0 count: 16526

==== split 0 ====

==== split 1 ====

==== split 2 ====

==== split 3 ====
xi>=0 count: 16944
xi>=0 count: 16697
xi>=0 count: 17524
xi>=0 count: 16459

==== split 0 ====

==== split 1 ====

==== split 2 ====

==== split 3 ====
xi>=0 count: 17730
xi>=0 count: 16813
xi>=0 count: 18249
xi>=0 count: 16462

==== split 0 ====

==== split 1 ====

==== split 2 ====

==== split 3 ====
xi>=0 count:

In [12]:
NOISE_dir='./noise/'
ATTACK_dir=Noise_dir='./attack/'

In [13]:
select_list = evalu_select(layer_list, y_train, eva_w, eva_b, pred_model=pred_model, save_dir="D:/Data_3/ViT-8/layer_cache/base")

[3 0 2 ... 3 2 2]
[3 0 2 ... 3 2 2]
[3 0 2 ... 3 2 2]
[3 0 2 ... 3 2 2]


In [14]:
base = evalu_stream_main_selected(layer_list, y_train, eva_w, eva_b, select_list, save_dir="D:/Data_3/ViT-8/layer_cache/base")
base_final = per_group_ovr_accuracy(model, x_train, y_train, select_list)
base = np.concatenate((base,base_final.reshape(1,4)),axis=0)

Layer 0
accuracy: [0.55177225 0.69829211 0.6834291  0.54420718]
Layer 1
accuracy: [0.61007866 0.72057939 0.71670122 0.57272242]
Layer 2
accuracy: [0.5504283  0.718375   0.72130302 0.57937587]
Layer 3
accuracy: [0.54384911 0.73261079 0.74846731 0.60815897]
Layer 4
accuracy: [0.649168   0.77549472 0.79241174 0.61630643]
Layer 5
accuracy: [0.69712233 0.77189808 0.8326298  0.61512524]
Layer 6
accuracy: [0.77496677 0.79816447 0.85816748 0.69774065]
Layer 7
accuracy: [0.88629939 0.93776478 0.94270193 0.93317317]
Layer 8
accuracy: [0.96625432 0.97470615 0.98048079 0.97117391]
Layer 9
accuracy: [0.94505672 0.95721126 0.93306134 0.94199563]


In [15]:
base

array([[0.55177225, 0.69829211, 0.6834291 , 0.54420718],
       [0.61007866, 0.72057939, 0.71670122, 0.57272242],
       [0.5504283 , 0.718375  , 0.72130302, 0.57937587],
       [0.54384911, 0.73261079, 0.74846731, 0.60815897],
       [0.649168  , 0.77549472, 0.79241174, 0.61630643],
       [0.69712233, 0.77189808, 0.8326298 , 0.61512524],
       [0.77496677, 0.79816447, 0.85816748, 0.69774065],
       [0.88629939, 0.93776478, 0.94270193, 0.93317317],
       [0.96625432, 0.97470615, 0.98048079, 0.97117391],
       [0.94505672, 0.95721126, 0.93306134, 0.94199563],
       [1.        , 1.        , 1.        , 1.        ]])

In [16]:
gauss = np.zeros((len(layer_list),4))
for i in range(10):
    gauss += evalu_stream_main_selected(layer_list, y_train, eva_w, eva_b, select_list, save_dir = "D:/Data_3/ViT-8/layer_cache/gauss/"+str(i))
gauss = gauss/10
gauss_final = np.zeros(4)
for i in range(10):
    DIR = NOISE_dir + str(i) + '/gauss.npy'
    x_noise = np.load(DIR)
    gauss_final += per_group_ovr_accuracy(model, x_noise, y_train, select_list)
gauss_final = gauss_final/10
gauss = np.concatenate((gauss,gauss_final.reshape(1,4)),axis=0)

Layer 0
accuracy: [0.65058181 0.72632927 0.71889155 0.61931525]
Layer 1
accuracy: [0.71839358 0.74365893 0.7460716  0.68974222]
Layer 2
accuracy: [0.68100222 0.74426329 0.73979896 0.69921168]
Layer 3
accuracy: [0.56906979 0.73503922 0.73772204 0.64534332]
Layer 4
accuracy: [0.63424557 0.75893712 0.76322251 0.62135223]
Layer 5
accuracy: [0.66665627 0.75792755 0.79094493 0.62098402]
Layer 6
accuracy: [0.70259302 0.77155654 0.81404397 0.64276663]
Layer 7
accuracy: [0.79283668 0.85934956 0.86589158 0.83591876]
Layer 8
accuracy: [0.84785486 0.85403341 0.88050527 0.85317717]
Layer 9
accuracy: [0.84139199 0.86883875 0.85303456 0.8492184 ]
Layer 0
accuracy: [0.65108694 0.72527551 0.72009108 0.62086567]
Layer 1
accuracy: [0.71612798 0.74389873 0.74505489 0.69139823]
Layer 2
accuracy: [0.67466496 0.74541123 0.73753076 0.70441416]
Layer 3
accuracy: [0.570896   0.7366997  0.7344325  0.65231643]
Layer 4
accuracy: [0.63721464 0.75925534 0.76317758 0.6221374 ]
Layer 5
accuracy: [0.6589552  0.76017318

In [17]:
gauss

array([[0.6499016 , 0.72691909, 0.72082776, 0.62093484],
       [0.71814191, 0.7439904 , 0.74592482, 0.69258745],
       [0.67852874, 0.74566112, 0.73829156, 0.70201059],
       [0.57374852, 0.73780806, 0.73656116, 0.64767501],
       [0.63327737, 0.75960836, 0.76416066, 0.6223071 ],
       [0.66098579, 0.75962571, 0.79117141, 0.61863796],
       [0.70110155, 0.77482882, 0.81090521, 0.64210923],
       [0.79486405, 0.85446803, 0.8546132 , 0.83448859],
       [0.85267782, 0.8571716 , 0.87058785, 0.84839173],
       [0.84037256, 0.87003496, 0.8488389 , 0.84500548],
       [0.91659999, 0.90947499, 0.92359999, 0.89732499]])

In [18]:
salt = np.zeros((len(layer_list),4))
for i in range(10):
    salt += evalu_stream_main_selected(layer_list, y_train, eva_w, eva_b, select_list, save_dir = "D:/Data_3/ViT-8/layer_cache/salt/"+str(i))
salt = salt/10
salt_final = np.zeros(4)
for i in range(10):
    DIR = NOISE_dir + str(i) + '/salt.npy'
    x_noise = np.load(DIR)
    salt_final += per_group_ovr_accuracy(model, x_noise, y_train, select_list)
salt_final = salt_final/10
salt = np.concatenate((salt,salt_final.reshape(1,4)),axis=0)

Layer 0
accuracy: [0.6404156  0.72009278 0.71710907 0.59408735]
Layer 1
accuracy: [0.70945166 0.7447401  0.74457809 0.66730575]
Layer 2
accuracy: [0.66199717 0.75021464 0.74550787 0.67926125]
Layer 3
accuracy: [0.58589662 0.74594431 0.74790126 0.63880331]
Layer 4
accuracy: [0.66722947 0.76904343 0.78112805 0.62148334]
Layer 5
accuracy: [0.6936126  0.76730721 0.81363914 0.60995592]
Layer 6
accuracy: [0.74456917 0.78812148 0.83538308 0.66097443]
Layer 7
accuracy: [0.82775167 0.88646133 0.891423   0.8667503 ]
Layer 8
accuracy: [0.9003519  0.90013713 0.91596453 0.89309819]
Layer 9
accuracy: [0.86440629 0.89729304 0.86727699 0.87086744]
Layer 0
accuracy: [0.63760966 0.71641592 0.71513517 0.59277134]
Layer 1
accuracy: [0.7086709  0.74551384 0.74783051 0.66807358]
Layer 2
accuracy: [0.66641943 0.75052167 0.74651125 0.67574686]
Layer 3
accuracy: [0.58384804 0.74239983 0.74545826 0.6384043 ]
Layer 4
accuracy: [0.66506865 0.77047708 0.7811248  0.62299937]
Layer 5
accuracy: [0.70062378 0.77023287

In [19]:
salt

array([[0.6382449 , 0.7186335 , 0.71539108, 0.59246905],
       [0.70930485, 0.74368092, 0.74700378, 0.66892403],
       [0.6652632 , 0.74733208, 0.74422328, 0.67815357],
       [0.58337407, 0.74283996, 0.74581407, 0.63968102],
       [0.66205216, 0.77033327, 0.77796124, 0.62189536],
       [0.69627879, 0.76897594, 0.81155118, 0.61357185],
       [0.74715787, 0.78609765, 0.83161116, 0.65602134],
       [0.83266487, 0.8857255 , 0.89176332, 0.87121668],
       [0.89911843, 0.90606575, 0.9132712 , 0.89906016],
       [0.86567644, 0.89908414, 0.86792159, 0.8755718 ],
       [0.954525  , 0.946925  , 0.95932501, 0.941825  ]])

In [20]:
SAVE_FILE='e-ViT-8.pkl'

In [21]:
progress = {"base": base, "gauss": gauss,"salt": salt}
with open(SAVE_FILE, "wb") as f:
    pickle.dump(progress, f)

In [22]:
def stats_minmax_from_matrix_dict(matrix_dict, check_num=3): 
    """ 
    Input: 
        matrix_dict: {name: np.ndarray(m, n)}, a dictionary containing 3 matrices 
    Output: 
        stats: { 
            name: { 
                "mean": (m,), 
                "min":  (m,), 
                "max":  (m,) 
            }, ... 
        } 
    For each component i, statistics are computed along axis=1 (over n samples): 
        mean[i] = mean(X[i, :]) 
        min[i]  = min(X[i, :]) 
        max[i]  = max(X[i, :]) 
    """ 
    if check_num is not None and len(matrix_dict) != check_num: 
        raise ValueError(f"Expected {check_num} matrices, but received {len(matrix_dict)}") 
 
    # shape consistency 
    shapes = [np.asarray(v).shape for v in matrix_dict.values()] 
    if len(set(shapes)) != 1: 
        raise ValueError(f"Inconsistent matrix shapes: {shapes}") 
 
    stats = {} 
    for name, X in matrix_dict.items(): 
        X = np.asarray(X) 
        stats[name] = { 
            "mean": X.mean(axis=1), 
            "std":  X.std(axis=1), 
            "min":  X.min(axis=1), 
            "max":  X.max(axis=1), 
        } 
    return stats 

In [24]:
mean_var = stats_minmax_from_matrix_dict(progress)

ValueError: 期望 6 个矩阵，但收到 3 个

In [ ]:
mean_var